# E3 — FAID-Style Fusion-Aware Contrastive XLM-R

**Goal:** three-class classification (`human` / `ai` / `fusion`) with two auxiliary signals on top of cross-entropy:
1. **AI-ratio regression head** — predicts a scalar in `[0, 1]` where `human=0.0`, `fusion=0.5`, `ai=1.0`.
2. **Fusion-aware contrastive loss (`L_FAID`)** — for each `fusion` sample, pull it toward its nearest `ai` neighbor in the batch and push it away from its nearest `human` neighbor.

`L = L_CE + 0.2 * L_aux + 0.1 * L_FAID`

Reference: E1 (Chunked XLM-R). This is compared against the head+tail XLM-R baseline on the same fixed val split (`val_fixed.csv`).

In [1]:
import os, json, time, random, re, math
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import TrainerCallback
from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)

# ----------------------------- config --------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

LABELS   = ["human", "ai", "fusion"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
AI_RATIO = {"human": 0.0, "fusion": 0.5, "ai": 1.0}

INPUT   = "/kaggle/input/datasets/suchok/base-train-data"
BASE    = "/kaggle/working"
EXP_DIR = f"{BASE}/experiments/e3_faid_contrastive"
os.makedirs(EXP_DIR, exist_ok=True)

MODEL_NAME = "xlm-roberta-base"
MAX_LEN    = 512

# Loss weights
AUX_WEIGHT = 0.2
LAM_FAID   = 0.1
TAU        = 0.07

# Training
TRAIN_BS   = 8
GRAD_ACC   = 2         # effective batch = 16
EPOCHS     = 2
LR_ENCODER = 2e-5
LR_HEAD    = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## Loading

In [2]:
sub40 = pd.read_csv(f"{INPUT}/train_transformer_40k.csv")
val   = pd.read_csv(f"{INPUT}/val_fixed.csv")

sub40["label_id"] = sub40["label"].map(LABEL2ID)
val["label_id"]   = val["label"].map(LABEL2ID)

# Sanity
assert sub40["label_id"].notna().all() and val["label_id"].notna().all()

print(f"Train: {len(sub40):,} | Val: {len(val):,}")
print("Train class dist:\n", sub40["label"].value_counts())
print("Val   class dist:\n", val["label"].value_counts())

Train: 40,000 | Val: 40,853
Train class dist:
 label
ai        17909
fusion    12290
human      9801
Name: count, dtype: int64
Val   class dist:
 label
ai        18291
fusion    12552
human     10010
Name: count, dtype: int64


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    """Expects a dataframe with `text` and `label_id` columns."""
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.texts  = df["text"].astype(str).tolist()
        self.labels = df["label_id"].astype(int).tolist()
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_attention_mask=True,
        )
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        # dummy sty tensor kept for interface compatibility (unused)
        item["sty"] = torch.tensor(0, dtype=torch.long)
        return item

@dataclass
class FusionCollator:
    """Dynamic padding for input_ids / attention_mask."""
    tokenizer: object
    def __call__(self, features):
        # pull labels out before padding
        labels = torch.stack([f.pop("labels") for f in features])
        sty    = torch.stack([f.pop("sty")    for f in features])
        batch  = self.tokenizer.pad(
            features, padding=True, return_tensors="pt"
        )
        batch["labels"] = labels
        batch["sty"]    = sty
        return batch

train_ds = TextDataset(sub40, tokenizer)
val_ds   = TextDataset(val,   tokenizer)
collator = FusionCollator(tokenizer=tokenizer)

print(f"Train size={len(train_ds):,} | Val size={len(val_ds):,}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Train size=40,000 | Val size=40,853


In [4]:
class FusionAwareXLMR(nn.Module):
    """
    XLM-R base + (3-way classifier head) + (1-D AI-ratio regression head).
    Returns dict with logits, aux_pred, cls_emb so the custom loss can use them.
    """
    def __init__(self, model_name=MODEL_NAME, num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size         # 768 for xlm-r-base
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)  # human / ai / fusion
        self.aux_head   = nn.Linear(hidden, 1)           # AI-ratio regression

    def forward(self, input_ids, attention_mask, labels=None,
                sty=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]             # [B, 768]
        cls_d = self.dropout(cls)

        logits   = self.classifier(cls_d)                # [B, 3]
        aux_pred = self.aux_head(cls_d).squeeze(-1)      # [B]

        return {
            "logits":   logits,
            "aux_pred": aux_pred,
            "cls_emb":  cls,                              # un-dropped, for contrastive
            "labels":   labels,
        }


def fusion_aware_contrastive_loss(cls_emb, labels, tau=TAU, eps=1e-8):
    """
    For each fusion sample in the batch:
      positive = most cosine-similar *ai* sample
      negative = most cosine-similar *human* sample
    InfoNCE over the two similarities.
    Returns mean loss over fusion samples (0.0 if batch lacks any class).
    """
    if cls_emb is None or cls_emb.size(0) == 0:
        return cls_emb.new_zeros(())
    # Guard: need at least 1 fusion, 1 ai, 1 human
    have = set(labels.tolist())
    if not {LABEL2ID["fusion"], LABEL2ID["ai"], LABEL2ID["human"]}.issubset(have):
        return cls_emb.new_zeros(())

    # Cosine similarity matrix [B, B]
    z = F.normalize(cls_emb.float(), dim=-1, eps=eps)
    sim = z @ z.t()                                       # in [-1, 1]

    fusion_idx = (labels == LABEL2ID["fusion"]).nonzero(as_tuple=True)[0]
    ai_idx     = (labels == LABEL2ID["ai"]).nonzero(as_tuple=True)[0]
    human_idx  = (labels == LABEL2ID["human"]).nonzero(as_tuple=True)[0]

    losses = []
    for fi in fusion_idx:
        # similarities of this fusion sample to all ai / human samples
        sim_ai    = sim[fi, ai_idx]                       # [n_ai]
        sim_human = sim[fi, human_idx]                    # [n_human]

        pos_sim = sim_ai.max()                            # nearest ai
        neg_sim = sim_human.max()                         # nearest human (hard neg)

        logits = torch.stack([pos_sim, neg_sim]) / tau    # [2]
        losses.append(-F.log_softmax(logits, dim=0)[0])   # -log p(pos)

    if not losses:
        return cls_emb.new_zeros(())
    return torch.stack(losses).mean()


class FusionTrainer(Trainer):
    """Trainer with the composite loss (CE + aux + FAID contrastive)."""
    def __init__(self, *args, ce_weight=None, aux_weight=AUX_WEIGHT,
                 lam_faid=LAM_FAID, tau=TAU, **kwargs):
        super().__init__(*args, **kwargs)
        self.ce_weight  = ce_weight
        self.aux_weight = aux_weight
        self.lam_faid   = lam_faid
        self.tau        = tau

    def compute_loss(self, model, inputs, return_outputs=False,
                     num_items_in_batch=None):
        labels = inputs["labels"]
        sty    = inputs.pop("sty", None)

        outputs = model(**inputs, sty=sty)
        logits   = outputs["logits"]
        aux_pred = outputs["aux_pred"]
        cls_emb  = outputs["cls_emb"]

        # 1) cross-entropy with class weights
        ce = F.cross_entropy(logits, labels, weight=self.ce_weight)

        # 2) auxiliary AI-ratio regression
        ai_targets = torch.tensor(
            [AI_RATIO[ID2LABEL[int(l)]] for l in labels],
            dtype=torch.float, device=labels.device
        )
        aux = F.mse_loss(aux_pred, ai_targets)

        # 3) fusion-aware contrastive loss
        faid = fusion_aware_contrastive_loss(cls_emb, labels, tau=self.tau)

        loss = ce + self.aux_weight * aux + self.lam_faid * faid
        loss = torch.nan_to_num(loss, nan=0.0, posinf=1e6, neginf=-1e6)

        # Expose components for logging
        self._last_loss_parts = dict(
            ce=float(ce.detach()),
            aux=float(aux.detach()),
            faid=float(faid.detach()),
        )

        if return_outputs:
            return loss, {**outputs, "labels": labels}
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only,
                        ignore_keys=None):
        with torch.no_grad():
            inputs = {k: v for k, v in inputs.items()}
            labels = inputs["labels"]
            sty    = inputs.pop("sty", None)
            outputs = model(**inputs, sty=sty)
            logits  = outputs["logits"]
        return (None, logits.detach(), labels.detach())

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro = f1_score(labels, preds, average="macro")
    p, r, f, _ = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0, 1, 2], zero_division=0
    )
    return {
        "macro_f1": macro,
        "f1_human": f[0], "f1_ai": f[1], "f1_fusion": f[2],
        "p_fusion": p[2], "r_fusion": r[2],
    }


def build_optimizer(model, lr_enc=LR_ENCODER, lr_head=LR_HEAD, wd=0.01):
    """Discriminative LR: encoder=2e-5 (no decay on bias/LN), heads=1e-4."""
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    encoder = model.encoder
    head_params = list(model.classifier.parameters()) + \
                  list(model.aux_head.parameters())
    head_ids = {id(p) for p in head_params}

    enc_decay, enc_nodecay = [], []
    for n, p in encoder.named_parameters():
        if not p.requires_grad:
            continue
        (enc_nodecay if any(nd in n for nd in no_decay) else enc_decay).append(p)

    groups = [
        {"params": enc_decay,   "lr": lr_enc,  "weight_decay": wd},
        {"params": enc_nodecay, "lr": lr_enc,  "weight_decay": 0.0},
        {"params": head_params, "lr": lr_head, "weight_decay": wd},
    ]
    return torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-6)


class LiveLogCallback(TrainerCallback):
    """
    Prints training progress every `every_n` optimizer steps.
    Uses on_step_end so it fires on the actual global_step boundary
    (independent of logging_steps), then reads the latest log line for
    loss / lr when available.
    """
    def __init__(self, every_n=50):
        self.every_n = every_n
        self.t0 = time.time()
        self._first_step_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()
        self._first_step_time = None
        print(f"[train] begin | max_steps={state.max_steps} | "
              f"eff_batch={args.per_device_train_batch_size * args.gradient_accumulation_steps}",
              flush=True)
        return control

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == 0:
            return control
        if self._first_step_time is None:
            self._first_step_time = time.time()
        if state.global_step % self.every_n != 0:
            return control

        elapsed = time.time() - self._first_step_time
        # step count that actually required time (avoid div by 0)
        steps_measured = max(state.global_step - 1, 1)
        sps = steps_measured / max(elapsed, 1e-6)
        remaining = max(state.max_steps - state.global_step, 0)
        eta_min = (remaining / sps) / 60.0 if sps > 0 else 0.0

        loss_str = ""
        if state.log_history:
            last = state.log_history[-1]
            if "loss" in last:
                loss_str = f" loss={last['loss']:.4f}"
            if "learning_rate" in last:
                loss_str += f" lr={last['learning_rate']:.2e}"
            if "epoch" in last:
                loss_str += f" ep={last['epoch']:.2f}"

        print(f"[step {state.global_step:>5}/{state.max_steps}]"
              f"{loss_str} | {sps:.2f} it/s | ETA {eta_min:.1f} min",
              flush=True)
        return control


class LossPartsCallback(TrainerCallback):
    """
    Prints the CE / aux / FAID loss components from the trainer every `every_n` steps.
    Reads `trainer._last_loss_parts` (set inside FusionTrainer.compute_loss).
    """
    def __init__(self, trainer_ref, every_n=50):
        self.trainer_ref = trainer_ref
        self.every_n = every_n

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == 0 or state.global_step % self.every_n != 0:
            return control
        parts = getattr(self.trainer_ref, "_last_loss_parts", None)
        if parts:
            print(f"    └─ ce={parts['ce']:.4f}  aux={parts['aux']:.4f}  "
                  f"faid={parts['faid']:.4f}", flush=True)
        return control

In [6]:
# Inverse-frequency weights over the 40k subsample
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=sub40["label_id"].values,
)
ce_weight = torch.tensor(cw, dtype=torch.float, device=device)
print("CE class weights [human, ai, fusion]:", ce_weight.tolist())

training_args = TrainingArguments(
    output_dir=EXP_DIR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    learning_rate=LR_ENCODER,       # overridden by build_optimizer
    weight_decay=0.01,
    warmup_steps=0.06,              # fraction (<=1.0) -> Trainer interprets as ratio
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    disable_tqdm=True,
)

model = FusionAwareXLMR(MODEL_NAME).to(device)
optimizer = build_optimizer(model)

trainer = FusionTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    ce_weight=ce_weight,
    aux_weight=AUX_WEIGHT,
    lam_faid=LAM_FAID,
    tau=TAU,
)

# Progress callbacks — now proper TrainerCallback subclasses
trainer.add_callback(LiveLogCallback(every_n=50))
trainer.add_callback(LossPartsCallback(trainer_ref=trainer, every_n=50))

print("Effective batch size:", TRAIN_BS * GRAD_ACC)
print("Starting training...")

train_result = trainer.train()
print("\n=== Training finished ===")
print(train_result.metrics)

CE class weights [human, ai, fusion]: [1.360405445098877, 0.7445046305656433, 1.0848928689956665]


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Effective batch size: 16
Starting training...
[train] begin | max_steps=2500 | eff_batch=16
{'loss': '2.72', 'grad_norm': '14.19', 'learning_rate': '3.2e-06', 'epoch': '0.02'}
[step    50/2500] loss=2.7201 lr=3.20e-06 ep=0.02 | 0.45 it/s | ETA 91.6 min
    └─ ce=1.1810  aux=0.1875  faid=0.5745
{'loss': '2.542', 'grad_norm': '15.13', 'learning_rate': '6.533e-06', 'epoch': '0.04'}
{'loss': '2.459', 'grad_norm': '12.71', 'learning_rate': '9.867e-06', 'epoch': '0.06'}
[step   100/2500] loss=2.4592 lr=9.87e-06 ep=0.06 | 0.44 it/s | ETA 91.0 min
    └─ ce=0.9123  aux=0.1075  faid=0.5833
{'loss': '2.321', 'grad_norm': '18.49', 'learning_rate': '1.32e-05', 'epoch': '0.08'}
{'loss': '2.034', 'grad_norm': '17.23', 'learning_rate': '1.653e-05', 'epoch': '0.1'}
[step   150/2500] loss=2.0337 lr=1.65e-05 ep=0.10 | 0.44 it/s | ETA 89.8 min
    └─ ce=0.6872  aux=0.0870  faid=0.4420
{'loss': '1.745', 'grad_norm': '17.26', 'learning_rate': '1.987e-05', 'epoch': '0.12'}
{'loss': '1.638', 'grad_norm': '32

In [7]:
# Full evaluation on the fixed val split
preds_out = trainer.predict(val_ds)
logits    = preds_out.predictions
labels_np = preds_out.label_ids
preds_np  = np.argmax(logits, axis=-1)

probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

macro = f1_score(labels_np, preds_np, average="macro")
print(f"\nVal macro-F1: {macro:.4f}\n")
print("Per-class report:")
print(classification_report(
    labels_np, preds_np,
    target_names=LABELS, digits=4, zero_division=0,
))
cm = confusion_matrix(labels_np, preds_np, labels=[0, 1, 2])
print("Confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS))

# Save artifacts
np.save(os.path.join(EXP_DIR, "preds.npy"), preds_np)
np.save(os.path.join(EXP_DIR, "probs.npy"), probs)
np.save(os.path.join(EXP_DIR, "logits.npy"), logits)
np.save(os.path.join(EXP_DIR, "labels.npy"), labels_np)
pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(
    os.path.join(EXP_DIR, "confusion_matrix.csv")
)

# Full custom state dict (encoder + both heads)
torch.save(model.state_dict(), os.path.join(EXP_DIR, "model_state_dict.pt"))

# HF-format encoder + heads
model.encoder.save_pretrained(os.path.join(EXP_DIR, "hf_encoder"))
tokenizer.save_pretrained(os.path.join(EXP_DIR, "hf_encoder"))
torch.save(
    {"classifier": model.classifier.state_dict(),
     "aux_head":   model.aux_head.state_dict()},
    os.path.join(EXP_DIR, "heads.pt"),
)

# Metrics summary
metrics = dict(
    macro_f1=float(macro),
    per_class=dict(zip(LABELS,
        precision_recall_fscore_support(
            labels_np, preds_np, average=None, labels=[0,1,2],
            zero_division=0,
        )[2].tolist())),
    config=dict(
        model_name=MODEL_NAME, max_len=MAX_LEN,
        ce_weight=ce_weight.tolist(),
        aux_weight=AUX_WEIGHT, lam_faid=LAM_FAID, tau=TAU,
        train_bs=TRAIN_BS, grad_acc=GRAD_ACC, epochs=EPOCHS,
        lr_encoder=LR_ENCODER, lr_head=LR_HEAD, seed=SEED,
        train_rows=len(sub40), val_rows=len(val),
    ),
    train_metrics=train_result.metrics,
)
with open(os.path.join(EXP_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nArtifacts written to: {EXP_DIR}")
print(os.listdir(EXP_DIR))


Val macro-F1: 0.8601

Per-class report:
              precision    recall  f1-score   support

       human     0.7752    0.7783    0.7768     10010
          ai     0.9853    0.9948    0.9900     18291
      fusion     0.8206    0.8066    0.8135     12552

    accuracy                         0.8839     40853
   macro avg     0.8604    0.8599    0.8601     40853
weighted avg     0.8832    0.8839    0.8835     40853

Confusion matrix (rows=true, cols=pred):
        human     ai  fusion
human    7791     72    2147
ai         30  18195      66
fusion   2229    199   10124


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Artifacts written to: /kaggle/working/experiments/e3_faid_contrastive
['heads.pt', 'probs.npy', 'labels.npy', 'hf_encoder', 'preds.npy', 'metrics.json', 'checkpoint-1250', 'model_state_dict.pt', 'checkpoint-2500', 'confusion_matrix.csv', 'logits.npy']


In [8]:
# Fill these in with the head+tail XLM-R baseline and E1 numbers from your logs
references = {
    "head+tail XLM-R (baseline)": {"macro_f1": None, "f1_fusion": None},
    "E1 chunked XLM-R":           {"macro_f1": None, "f1_fusion": None},
    "E3 FAID (this run)":         {"macro_f1": float(macro),
                                    "f1_fusion": metrics["per_class"]["fusion"]},
}
pd.DataFrame(references).T

,macro_f1,f1_fusion
head+tail XLM-R (baseline),None,None
E1 chunked XLM-R,None,None
E3 FAID (this run),0.860106,0.813532
